# Ordered Logistic Regression Results: Adoption Predictors in Rangeland Management (FAIR^2) Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library, with all data elements referenced by their `@id` as per Croissant best practices.

### Dataset Source
The dataset is defined by a Croissant schema available at:
```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Instantiate and load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset loaded: {metadata.name}\n")
print(f"Summary: {metadata.description}\n")
print(f"Published: {metadata.datePublished}; Version: {metadata.version}")
print(f"License: {metadata.license}\nIdentifier: {metadata.identifier}")

## 2. Data Overview
Review all available record sets, fields, and their `@id` values defined in the Croissant schema. (References here are unique via the `@id`.)

In [ ]:
# List all record set @id values and their fields
from mlcroissant.structs.metadata import RecordSet

# Explore record sets
record_set_objs = list(metadata.record_sets)

if len(record_set_objs) == 0:
    print("No record sets are defined in the top-level Croissant metadata.")
else:
    for rs in record_set_objs:
        print(f"Record set: {rs.id}")
        # List fields
        if hasattr(rs, 'fields'):
            for fld in rs.fields:
                print(f"  Field: {fld.id}")
                if hasattr(fld, 'columns') and fld.columns:
                    for col in fld.columns:
                        print(f"    Column: {col.id}")
        print("")
    print(f"Total record sets: {len(record_set_objs)}")

# In this dataset's Croissant JSON, the recordSet is likely filled dynamically; mlcroissant may parse downloads and discover their record sets and fields, even if not at the top-level. Let's enumerate available record_sets by parsing from the schema if not defined in the 'recordSet' key.
if len(record_set_objs) == 0:
    print("mlcroissant did not parse any explicit record sets from metadata.\nTry invoking dataset.record_sets for inferred sets (for example, from distributions):")
    rs_ids = dataset.record_sets
    print("Inferred record sets:")
    for rsid in rs_ids:
        print(f" - {rsid}")

## 3. Data Extraction
Load data from available record sets into Pandas DataFrames using their `@id`. All tables and fields are referenced only by `@id`.

In [ ]:
# List available record set @ids:
record_set_ids = dataset.record_sets
print(f"Record sets detected (by @id): {record_set_ids}\n")

# Load all record sets into pandas DataFrames
dataframes = {}

for rsid in record_set_ids:
    # Each record set can be loaded with records(record_set=...) and is referenced by its @id
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"Loaded {len(df)} records from record set @id: {rsid}")

# Preview the columns of each DataFrame
for rsid, df in dataframes.items():
    print(f"\nRecord set @id: {rsid}")
    print(f"Columns (@id): {list(df.columns)}")
    display(df.head())  # Will show the first 5 records for each table

## 4. Exploratory Data Analysis (EDA)
Process, filter, and transform data with field references always by `@id` only.

> _**Note**: Replace the example variable/field @id's below with those relevant to your exploration, as discovered in the overview and extraction above._

In [ ]:
# --- Example EDA using first available record set and a numeric field ---

# Select a record set for analysis
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    print(f"\nUsing record set @id: {record_set_id}")

    # Find candidate numeric fields (by inspecting data)
    numeric_candidate = None
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_candidate = c
            break
    if not numeric_candidate:
        # Try to coerce non-numeric columns
        for c in df.columns:
            coerced = pd.to_numeric(df[c], errors='coerce')
            num_nulls = coerced.isnull().sum()
            if num_nulls < len(df) / 2:
                df[c] = coerced
                numeric_candidate = c
                print(f"Coerced column {c} to numeric (nulls: {num_nulls}).")
                break
    
    if numeric_candidate:
        numeric_field_id = numeric_candidate
        print(f"Using numeric field @id: {numeric_field_id}\n")
        # Filtering
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records in '@id' {record_set_id} where {numeric_field_id} > {threshold}\n")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping: find a suitable group-by candidate (categorical field)
        group_field_id = None
        # Find first string or object dtype column other than numeric
        for c in df.columns:
            if c != numeric_field_id and pd.api.types.is_object_dtype(df[c]):
                group_field_id = c
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable field found for grouping.")
    else:
        print("No suitable numeric field found for EDA in this record set.")
else:
    print("No record sets found; cannot proceed with EDA.")

## 5. Visualization
Visualize the distribution of the selected (numeric) field and its grouping.

In [ ]:
import matplotlib.pyplot as plt

# Plot only if we have data and numeric field
if record_set_ids and 'numeric_field_id' in locals():
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    # Histogram
    df[numeric_field_id].hist(ax=ax[0], bins=20, color='skyblue', edgecolor='black')
    ax[0].set_title(f"Distribution of {numeric_field_id}")
    ax[0].set_xlabel(numeric_field_id)
    ax[0].set_ylabel('Count')

    # If grouping done, plot mean by group
    if 'grouped_df' in locals() and not grouped_df.empty:
        grouped_df.plot(kind='bar', ax=ax[1], legend=False, color='orange')
        ax[1].set_title(f"Mean {numeric_field_id} by {group_field_id}")
        ax[1].set_xlabel(group_field_id)
        ax[1].set_ylabel(f"Mean {numeric_field_id}")
    else:
        ax[1].set_visible(False)
    plt.tight_layout()
    plt.show()
else:
    print("Visualization not possible: record set or numeric field missing.")

## 6. Conclusion
In this notebook, we demonstrated how to load the FAIR^2 dataset as defined by its Croissant schema and explored its structure and contents programmatically using the `mlcroissant` library.

- We referenced all tables, fields, and columns using their Croissant `@id` identifiers.
- We loaded record sets dynamically and performed filtering, normalization, and aggregation using Pandas.
- Visual analysis provided insights into data distributions and groupwise statistics.

**For in-depth analysis or machine learning tasks, use the field @id's as discovered in your exploration, and refer to the Croissant schema documentation or dataset metadata for more details.**